In [13]:
# 1) 망가진 설치 제거
!rm -rf /usr/local/lib/ollama /usr/local/bin/ollama
!pkill ollama 2>/dev/null; sleep 2

In [14]:
# 2) zstd 확실히 깔고 재설치
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [15]:
# 3) 설치 제대로 됐나 확인 — 바이너리 존재 체크
!ls -la /usr/local/lib/ollama/ | grep llama-server

-rwxr-xr-x 1 root root 2727856 Jun 17 06:23 libllama-server-impl.so
-rwxr-xr-x 1 root root   15120 Jun 17 06:23 llama-server


In [16]:
# 기존 serve 죽이고 새로 띄우기
!pkill ollama 2>/dev/null
import time; time.sleep(3)

import subprocess
subprocess.Popen(["ollama","serve"], stdout=open("ollama.log","w"), stderr=subprocess.STDOUT)
time.sleep(8)

!ollama list   # 모델 확인

NAME              ID              SIZE      MODIFIED      
exaone3.5:7.8b    c7c4e3d1ca22    4.8 GB    2 minutes ago    
exaone3.5:2.4b    13644fc3d28e    1.6 GB    3 minutes ago    


In [9]:
# 2.4b 먼저
!ollama pull exaone3.5:2.4b
!ollama list   # 받아졌나 확인


NAME              ID              SIZE      MODIFIED               
exaone3.5:2.4b    13644fc3d28e    1.6 GB    Less than a second ago    


In [10]:
!ollama pull exaone3.5:7.8b

In [17]:
import ollama

def ask(model, system, user, n=3):
    outs = []
    for _ in range(n):
        r = ollama.chat(model=model,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            options={"temperature":0.7})
        outs.append(r["message"]["content"].strip())
    return outs

SYS = """너는 공공 SW사업 과업수행계획서를 평가하는 심사자다.
주어진 평가기준에 따라 계획서 내용을 판정하라.
판정은 충족/미흡/불가 중 하나. 출력 형식:
판정: {충족|미흡|불가}
코멘트: {한 문장, 어떤 근거요소가 있고 없는지}"""

T1 = """[평가기준] 특정 기술을 미사용하는 경우 그 근거가 구체적으로 제시되었는가?
[계획서] 본 사업은 데이터 이관 중심 사업으로 별도 성능요건이 없어 AWS를 사용하지 않는다."""

T2 = """[평가기준] 특정 기술을 미사용하는 경우 그 근거가 구체적으로 제시되었는가?
[계획서] 본 사업의 특성상 AWS를 사용하지 않는다."""

T3 = """[평가기준] 개인정보를 처리하는 경우 개인정보 영향평가(PIA) 계획이 포함되었는가?
[RFP] 학생 학적·성적 정보를 통합 관리하는 시스템을 구축한다.
[계획서] PIA: 해당없음"""

T4 = """[평가기준] 개인정보를 처리하는 경우 개인정보 영향평가(PIA) 계획이 포함되었는가?
[RFP] 학생 학적·성적 정보를 통합 관리하는 시스템을 구축한다.
[계획서] 학생 개인정보를 처리하므로 설계 단계에서 PIA를 수행하고 결과를 반영한다."""

tests = {"T1_법아닌출처":T1, "T2_동어반복":T2, "T3_조건부분기":T3, "T4_정상":T4}

for model in ["exaone3.5:2.4b", "exaone3.5:7.8b"]:
    print("\n" + "#"*60); print("MODEL:", model); print("#"*60)
    for name, q in tests.items():
        print(f"\n--- {name} ---")
        for i, o in enumerate(ask(model, SYS, q, n=3), 1):
            print(f"[{i}] {o}")


############################################################
MODEL: exaone3.5:2.4b
############################################################

--- T1_법아닌출처 ---
[1] 판정: **미흡**

코멘트: 계획서는 AWS 미사용의 이유를 단순히 "데이터 이관 중심 사업"이라는 일반적인 설명으로 제시하고 있어 구체적인 근거가 부족하다. 기술 선택의 배경이나 대안 기술의 적합성, 비용 대비 효과 등 구체적인 분석과 이유가 누락되어 있어 명확성과 설득력이 떨어진다.
[2] 판정: **미흡**

코멘트: 계획서는 AWS 미사용의 이유를 단순히 "데이터 이관 중심 사업으로 별도 성능요건이 없다"는 추상적인 근거로 제시하고 있어 구체적인 근거나 세부적인 분석이 부족하다. 성능 요건 외에 더 구체적인 기술적 또는 비즈니스적 이유가 누락되어 있어 명확성이 떨어진다.
[3] 판정: **미흡**

코멘트: 계획서는 AWS 미사용의 이유를 단순히 "데이터 이관 중심 사업으로 별도 성능요건이 없다"는 일반적인 설명으로 제시하고 있어 구체적인 근거나 분석이 부족하다. 기술 선택의 배경이나 대안 기술의 장단점 비교 등 더 세밀한 근거 제시가 필요하다.

--- T2_동어반복 ---
[1] **판정:** 미흡  
**코멘트:** 계획서는 AWS 미사용 이유를 단순히 언급하고 있지만, 구체적인 근거나 대안 기술의 선택 이유, 비용 효율성, 기술적 적합성 등을 명확히 제시하지 않아 미흡하다. 명확한 근거와 함께 대안 기술의 상세한 분석이 필요하다.
[2] 판정: **미흡**

코멘트: 계획서는 AWS 미사용의 이유를 단순히 언급하고 있지만, 구체적인 근거나 대안 기술의 선택 배경, 비용 분석, 프로젝트 요구 사항과의 일치성 등을 명확히 제시하지 않았습니다. 구체적인 이유와 그 근거가 더 필요합니다.
[3] 판정: **미흡**

코멘트: 계획서에서 AWS 미사용의 근거가

In [18]:
SYS2 = """너는 공공 SW사업 과업수행계획서를 평가하는 심사자다.
주어진 평가기준과 판정규칙에 따라 판정하라.

[판정규칙]
- 충족: 미사용에 대한 '이유'가 하나라도 구체적으로 제시됨 (사업특성/기술/규정 등 무엇이든)
- 미흡: 이유는 있으나 '특성상' '등' 같은 공허한 표현뿐
- 불가: 이유 없이 선언만 함

대안 분석, 비용 효과까지 요구하지 말 것. 이유의 '존재와 구체성'만 본다.

출력:
판정: {충족|미흡|불가}
코멘트: {한 문장}"""

T1 = """[평가기준] 특정 기술을 미사용하는 경우 그 근거가 구체적으로 제시되었는가?
[계획서] 본 사업은 데이터 이관 중심 사업으로 별도 성능요건이 없어 AWS를 사용하지 않는다."""

for o in ask("exaone3.5:2.4b", SYS2, T1, n=3):
    print(o, "\n")

판정: **미흡**

코멘트: "별도 성능요건이 없다"는 설명만으로는 충분한 구체적인 근거가 부족합니다. 기술 선택의 근거로 더 명확한 기준이나 대안 기술의 장단점 비교 등이 필요합니다. 

판정: **미흡**

코멘트: 이유가 제시되었으나, "별도 성능요건이 없어"라는 표현은 구체적인 기술적 이유나 AWS 미사용의 명확한 근거를 명확히 제시하지 못하고 있어 부족한 면이 있다. 보다 구체적인 기술적 이유나 사업 특성에 기반한 차별화가 필요하다. 

판정: **미흡**

코멘트: '특정 기술 미사용' 이유가 '성능요건 부재'로 제시되었으나, 구체적인 근거나 데이터 이관 과정에서 AWS의 성능 또는 적합성이 왜 다른 기술로 대체될 수 있는지에 대한 설명이 부족합니다. 더욱 구체적인 기술적 이유나 대안의 장단점 분석이 필요합니다. 



In [21]:
SYS3 = """너는 공공 SW사업 과업수행계획서 평가 심사자다.
판정은 충족/미흡/불가 중 하나.
출력: 판정: {충족|미흡|불가} / 코멘트: {한 문장}"""

FEWSHOT = """[예시1]
[계획서] 클라우드 보안 인증(CSAP) 미획득 상용 서비스라 도입하지 않는다.
판정: 충족
코멘트: 미도입 이유(CSAP 미획득)가 구체적으로 제시됨.

[예시2]
[계획서] 여러 사정을 고려하여 해당 기술은 도입하지 않는다.
판정: 미흡
코멘트: '여러 사정'이라는 공허한 표현뿐, 구체적 이유 없음.

[예시3]
[계획서] 해당 기술은 도입하지 않는다.
판정: 불가
코멘트: 이유 없이 선언만 함.

이제 아래를 같은 기준으로 판정하라. 예시처럼 '이유가 구체적으로 있으면 충족'이다.
[계획서] 본 사업은 데이터 이관 중심 사업으로 별도 성능요건이 없어 AWS를 사용하지 않는다.
판정:"""

for o in ask("exaone3.5:7.8b", SYS3, FEWSHOT, n=3):
    print(o, "\n")

판정: 충족
코멘트: 데이터 이관 중심 사업임을 명확히 하여 AWS 사용을 피한 구체적인 이유가 제시됨. 

판정: 충족
코멘트: 데이터 이관 중심 사업으로 인한 구체적인 이유 제시로 인해 AWS 도입의 불필요성이 설명됨. 

판정: 충족
코멘트: 데이터 이관 중심 사업의 특성상 별도의 성능 요구사항이 없다는 구체적인 이유가 명시되어 있어 이해가 가능하다. 



In [20]:
# 누가 봐도 명확한 충족본
T1_clear = """[평가기준] 특정 기술을 미사용하는 경우 그 근거가 구체적으로 제시되었는가?
[계획서] 본 사업은 교육청 망분리 지침에 따라 외부 퍼블릭 클라우드(AWS) 사용이 금지되어 있어 행정망 내부 G-클라우드를 사용한다."""

for o in ask("exaone3.5:2.4b", SYS, T1_clear, n=3):
    print(o, "\n")

판정: **충족**

코멘트: 계획서에서 기술 선택의 제약 조건인 외부 퍼블릭 클라우드 사용 금지 지침을 명확히 언급하고 있으며, 그에 따른 대안으로 내부 G-클라우드 사용을 구체적으로 제시하고 있어 근거가 충분히 제공되었다고 판단됩니다. 

판정: **충족**

코멘트: 계획서에서 외부 퍼블릭 클라우드인 AWS 사용이 금지된 구체적인 이유와 교육청 망분리 지침의 내용이 명시되어 있어, 기술 미사용 근거가 명확하게 제시되어 있습니다. 

판정: **충족**

코멘트: 계획서는 명확하게 외부 퍼블릭 클라우드 사용 금지 정책에 따른 대체 방안으로 G-클라우드를 선택한 이유를 제시하고 있어, 평가기준에 부합합니다. 구체적인 근거로 정책 준수와 내부 보안 강화를 언급하고 있습니다. 

